In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis, normaltest, zscore, spearmanr, pearsonr
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


In [3]:
df = pd.read_csv("ecommerce_dataset.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (5000, 15)


,transaction_id,customer_id,timestamp,product_category,product_price,quantity,discount_pct,customer_age,is_returning_customer,device_type,region,payment_method,total_price,final_price,date
0,1,4174,2023-01-01 00:00:00,Electronics,32.034680,5,5.0,NaN,False,Mobile,South,Credit Card,160.173401,152.164731,2023-01-01
1,2,4507,2023-01-01 08:00:00,Electronics,147.516914,1,10.0,21.0,False,Mobile,West,PayPal,147.516914,132.765223,2023-01-01
2,3,1860,2023-01-01 16:00:00,Clothing,89.375768,2,0.0,40.0,True,Mobile,West,Bank Transfer,178.751536,178.751536,2023-01-01
3,4,2294,2023-01-02 00:00:00,Clothing,244.525565,3,5.0,45.0,False,Mobile,West,Credit Card,733.576694,696.897859,2023-01-02
4,5,2130,2023-01-02 08:00:00,Electronics,272.411662,1,15.0,39.0,False,Mobile,East,PayPal,272.411662,231.549913,2023-01-02


In [ ]:
print(df.info())

memory_usage = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nTotal Memory Usage: {memory_usage:.2f} MB")

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   transaction_id         5000 non-null   int64  
 1   customer_id            5000 non-null   int64  
 2   timestamp              5000 non-null   str    
 3   product_category       5000 non-null   str    
 4   product_price          5000 non-null   float64
 5   quantity               5000 non-null   int64  
 6   discount_pct           4950 non-null   float64
 7   customer_age           4950 non-null   float64
 8   is_returning_customer  5000 non-null   bool   
 9   device_type            5000 non-null   str    
 10  region                 5000 non-null   str    
 11  payment_method         5000 non-null   str    
 12  total_price            5000 non-null   float64
 13  final_price            5000 non-null   float64
 14  date                   5000 non-null   str    
dtypes: bool(1), flo

In [5]:
missing_values = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum()/len(df))*100
})

missing_values = missing_values.sort_values(
    by='Missing_Percentage',
    ascending=False
)

overall_missing_percentage = (
    df.isnull().sum().sum() /
    (df.shape[0]*df.shape[1])
)*100

print("Overall Missing Percentage:",
      round(overall_missing_percentage,2),"%")

display(missing_values)

missing_values.to_csv(
    "data_quality_report.csv",
    index=True
)

Overall Missing Percentage: 0.13 %


,Missing_Count,Missing_Percentage
discount_pct,50,1.0
customer_age,50,1.0
customer_id,0,0.0
timestamp,0,0.0
product_category,0,0.0
product_price,0,0.0
transaction_id,0,0.0
quantity,0,0.0
is_returning_customer,0,0.0
device_type,0,0.0


In [6]:
duplicate_rows = df.duplicated().sum()

print("Duplicate Rows:", duplicate_rows)

duplicate_transaction_groups = 0

if 'transaction_id' in df.columns:
    duplicate_transaction_groups = (
        df.duplicated(subset=['transaction_id'])
        .sum()
    )

print("Duplicate Transaction Groups:",
      duplicate_transaction_groups)

Duplicate Rows: 0
Duplicate Transaction Groups: 0


In [7]:
dtype_report = pd.DataFrame({
    "Column": df.columns,
    "DataType": df.dtypes.astype(str),
    "Missing": df.isnull().sum()
})

display(dtype_report)

,Column,DataType,Missing
transaction_id,transaction_id,int64,0
customer_id,customer_id,int64,0
timestamp,timestamp,str,0
product_category,product_category,str,0
product_price,product_price,float64,0
quantity,quantity,int64,0
discount_pct,discount_pct,float64,50
customer_age,customer_age,float64,50
is_returning_customer,is_returning_customer,bool,0
device_type,device_type,str,0


In [8]:
numeric_cols = df.select_dtypes(
    include=np.number
).columns.tolist()

print(numeric_cols)

['transaction_id', 'customer_id', 'product_price', 'quantity', 'discount_pct', 'customer_age', 'total_price', 'final_price']


In [9]:
stats_profile = df[numeric_cols].describe().T

stats_profile['median'] = df[numeric_cols].median()

stats_profile = stats_profile[
    ['count','mean','std','min',
     '25%','median','50%',
     '75%','max']
]

display(stats_profile)

stats_profile.to_csv(
    "statistical_profile.csv"
)

,count,mean,std,min,25%,median,50%,75%,max
transaction_id,5000.0,2500.500000,1443.520003,1.000000,1250.750000,2500.500000,2500.500000,3750.250000,5000.000000
customer_id,5000.0,3008.979400,1147.760648,1000.000000,2008.750000,3000.500000,3000.500000,4006.000000,4999.000000
product_price,5000.0,118.521610,98.108931,20.005283,48.967381,87.880759,87.880759,157.079259,822.617824
quantity,5000.0,3.045000,1.438120,1.000000,2.000000,3.000000,3.000000,4.000000,10.000000
discount_pct,4950.0,4.876768,6.068825,0.000000,0.000000,0.000000,0.000000,10.000000,20.000000
customer_age,4950.0,37.907677,11.396337,18.000000,29.250000,38.000000,38.000000,46.000000,78.000000
total_price,5000.0,359.228978,357.609693,20.566814,122.138075,234.806908,234.806908,464.059637,3186.525446
final_price,5000.0,342.267620,343.454387,17.481792,115.825805,225.980904,225.980904,442.576383,3027.199174
